In [ ]:
import base64
import time
import pandas as pd
import os
import google.generativeai as genai
from IPython.display import display
from IPython.display import Markdown

# Gemini API Key - Set this as an environment variable or replace with your actual key
# set GEMINI_API_KEY in your environment

# Configure the Gemini API
genai.configure(api_key=os.environ.get("GEMINI_API_KEY", ""))

# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# Read CSV
questions_df = pd.read_csv("../../VLAT Questions.csv")
questions = questions_df.to_dict('records')
responses_data = []

for i, question in enumerate(questions, start=1):
    time.sleep(5)
    try:
        print(f"\nProcessing question {i}:")
        print(question)
        
        # Path to your image
        image_path = "../../Images/" + str(question.get('vis', '')) + ".png"
        
        # Question text
        question_text = question.get('question: ', '')
        question_options = question.get('option:', '')
        correct_ans = str(question.get('correct', '')).strip()
        
        print(f"Processing image: {image_path}")
        print(f"Question: {question_text}")
        print(f"Options: {question_options}")
        print(f"Correct answer: {correct_ans}")
        
        # Read the image as binary and create Gemini image part
        with open(image_path, "rb") as image_file:
            image_data = image_file.read()
        
        # Define the prompt
        prompt = ('I am about to show you a graph and ask you a multiple-choice question about that graph. \n\n' +
                 'Task 1: Data Extraction and Table Creation: First, explicitly list ALL numerical values you can identify on both axes, then create a structured table using markdown syntax that includes ALL data points you identified above with appropriate column headers with units. \n \n' +
                 'Task 2: Sort the data: Sort the data in descending order by the numerical values. \n \n' +
                 'Task 3: Data Verification and Error Handling: Double-check if your table matches ALL elements in the graph by comparing each value in your table with the graph and updating your table with correct values, verify the sorting is correct, and before proceeding, confirm all corrections have been made and use ONLY the corrected data for analysis. \n \n' +
                 'Task 4: Question Analysis: Using ONLY the verified data in your table, compare EACH value individually with the reference value, for "less than" comparisons mark ALL values that are even slightly below the reference, for "greater than" comparisons mark ALL values that are even slightly above the reference, and show each comparison on a new line. \n\n' +
                 'Provide your reasoning with specific references to table values. \n\n' +
                 'End with: "Correct Answer: ". Just write the value, nothing else. Do not write anything after this. \n\n' +
                 'Let\'s solve this step by step.' +
                 '\n\n' + question_text + " " + question_options)
        
        # Create Gemini content parts - removed as we're using the new API format
        
        # Add retry logic for any API errors
        max_retries = 3
        retry_delay = 20  # seconds
        
        for retry in range(max_retries):
            try:
                # Use the Gemini 2.0 Pro model
                model = genai.GenerativeModel('gemini-2.0-pro-exp-02-05')
                
                # Create parts list with text and image
                parts = [
                    {"text": prompt},
                    {"inline_data": {
                        "mime_type": "image/png",
                        "data": base64.b64encode(image_data).decode('utf-8')
                    }}
                ]
                
                # Configure generation parameters
                generation_config = {
                    "temperature": 0.0,  # Match your GPT temperature of 0.0
                    "max_output_tokens": 5000,
                }
                
                time_start = time.perf_counter()
                
                # Generate the response
                response = model.generate_content(
                    parts,
                    generation_config=generation_config
                )
                
                time_end = time.perf_counter()
                
                # If we get here, response was successful
                break
                
            except Exception as e:
                print(f"Error during API call (attempt {retry + 1}/{max_retries}):", str(e))
                if retry < max_retries - 1:
                    print(f"\nAPI error, waiting {retry_delay} seconds before retry {retry + 1}/{max_retries}")
                    time.sleep(retry_delay)
                    continue
                raise  # Re-raise the last exception if we've exhausted all retries
        
        try:
            # Extract the full response text
            full_response = response.text
            print("\nAPI Response:", full_response[:200] + "...")  # Debug print - first 200 chars
            
            # Extract the answer after "Correct Answer: "
            if "Correct Answer: " in full_response:
                gpt_answer = full_response.split("Correct Answer: ")[-1].strip()
                # Case-insensitive comparison after stripping whitespace
                is_correct = gpt_answer.strip().upper() == correct_ans.strip().upper()
            else:
                gpt_answer = "Error: No answer in correct format"
                is_correct = "N/A"
                
        except Exception as e:
            print(f"Error processing response: {str(e)}")
            gpt_answer = f"Error: {str(e)}"
            is_correct = "N/A"
        
        responses_data.append([gpt_answer, time_end-time_start, is_correct])
        print(f"\nAnswer: {gpt_answer}")
        print(f"Time taken: {time_end-time_start:.2f} seconds")
        print(f"Correct? {is_correct}")
        
        # Increase delay between requests to 15 seconds
        time.sleep(max(15 - (time_end-time_start), 0))
        
    except Exception as e:
        print(f"Error processing question {i}:", str(e))
        responses_data.append([f"Error: {str(e)}", 0, "N/A"])

# Create Results directory if it doesn't exist
results_dir = "./"
os.makedirs(results_dir, exist_ok=True)

# Save results
results_df = pd.DataFrame(responses_data, columns=['response', 'time', 'correct_bool'])
results_df.index = range(1, results_df.shape[0] + 1)
results_df.to_csv(results_dir + "Gemini_VLAT_" + str(int(time.time())) + ".csv", index_label="id")
print("\n*** Finished ***")